# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets by their @id and name
print("Available Record Sets (@ids):")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"  @id: {rs['@id']}, name: {rs.get('name', '[no name]')}")

# View fields within each record set
for rs in record_sets:
    print(f"\nFields in Record Set @id: {rs['@id']} ({rs.get('name', '[no name]')})")
    fields = rs.get('field', [])
    # 'field' could be dict or list
    if isinstance(fields, dict):
        fields = [fields]
    for fd in fields:
        # Each field is a dict or reference
        if isinstance(fd, dict):
            print(f"    - @id: {fd.get('@id', '[no id]')}, name: {fd.get('name', '[no name]')}, description: {fd.get('description', '[no description]')}")
        elif isinstance(fd, str):
            print(f"    - @id: {fd}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

# Extract data from each record set
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from Record Set @id: {record_set_id}")
        else:
            print(f"No records found for Record Set @id: {record_set_id}")
    except Exception as ex:
        print(f"Error reading records for {record_set_id}: {ex}")

# For demonstration, pick the first available record set with data
main_record_set_id = None
for rs_id in dataframes.keys():
    main_record_set_id = rs_id
    break

if main_record_set_id:
    print(f"\nColumns in {main_record_set_id} DataFrame:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No DataFrames extracted.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

if main_record_set_id:
    df = dataframes[main_record_set_id]

    # Try to find a numeric field for demo
    # Heuristically pick something with int/float that has >2 unique values
    numeric_field_id = None
    for col in df.columns:
        try:
            v = pd.to_numeric(df[col], errors='coerce')
            if v.notnull().sum() > 0 and v.nunique(dropna=True) > 2:
                numeric_field_id = col
                break
        except Exception:
            continue

    if numeric_field_id is not None:
        print(f"Selected numeric field for analysis: {numeric_field_id}")
        threshold = np.nanmedian(pd.to_numeric(df[numeric_field_id], errors='coerce'))

        # Filter records
        filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold} (median): {len(filtered_df)} records")
        
        # Normalization (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (
            pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
            - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
        ) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to find a group/categorical field with few unique values
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and df[col].nunique() < max(20, 0.5 * len(df)):
                group_field = col
                break
        if group_field:
            print(f"Grouping by field: {group_field}")
            # Convert numeric field as float
            filtered_df_num = filtered_df.copy()
            filtered_df_num[numeric_field_id] = pd.to_numeric(filtered_df_num[numeric_field_id], errors='coerce')
            grouped_df = filtered_df_num.groupby(group_field, dropna=False).mean(numeric_only=True)
            print(grouped_df[[numeric_field_id]].head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found in main record set for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if main_record_set_id and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    numeric_values = pd.to_numeric(df[numeric_field_id], errors='coerce').dropna()
    plt.hist(numeric_values, bins=30, color='skyblue', edgecolor='gray')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If group_field is found, do a boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        df_box = df[[group_field, numeric_field_id]].dropna()
        # Keep only categories with at least a few records
        value_counts = df_box[group_field].value_counts()
        keep = value_counts[value_counts >= 3].index
        df_box = df_box[df_box[group_field].isin(keep)]
        if len(keep) > 0:
            df_box[numeric_field_id] = pd.to_numeric(df_box[numeric_field_id], errors='coerce')
            df_box.boxplot(by=group_field, column=[numeric_field_id], rot=45, figsize=(12, 5))
            plt.title(f'{numeric_field_id} by {group_field}')
            plt.suptitle('')
            plt.xlabel(group_field)
            plt.ylabel(numeric_field_id)
            plt.show()
else:
    print("No numeric field available for plotting.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* With `mlcroissant`, we loaded the dataset schema and records for mass spectrometry analysis.
* We reviewed the available record sets and their fields by their unique `@id`s.
* Data extraction and DataFrame assembly were demonstrated for all record sets, with initial inspections of column names and data samples.
* Exploratory steps highlighted selecting and processing numeric fields (e.g., filtering, normalization, grouping).
* Visualizations such as histograms and boxplots can provide insight into the distributions and relationships in the data.

Further analysis can include more advanced modeling, deeper data cleaning, and publication of both workflow and results using the Croissant-powered FAIR data ecosystem.